###  Loading corpus

In [ ]:
!wget https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt

--2026-08-17 15:19:27--  https://raw.githubusercontent.com/karpathy/char-rnn/master/data/tinyshakespeare/input.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1115394 (1.1M) [text/plain]
Saving to: ‘input.txt’

input.txt           100%[===================>]   1.06M  --.-KB/s    in 0.05s   

2026-08-17 15:19:28 (20.9 MB/s) - ‘input.txt’ saved [1115394/1115394]



In [67]:
with open('input.txt', 'r', encoding='utf-8') as f:
    text = f.read()

In [68]:
print(f"length of dataset in char: {len(text)}")

length of dataset in char: 1115394


In [69]:
print(text[:1000])

First Citizen:
Before we proceed any further, hear me speak.

All:
Speak, speak.

First Citizen:
You are all resolved rather to die than to famish?

All:
Resolved. resolved.

First Citizen:
First, you know Caius Marcius is chief enemy to the people.

All:
We know't, we know't.

First Citizen:
Let us kill him, and we'll have corn at our own price.
Is't a verdict?

All:
No more talking on't; let it be done: away, away!

Second Citizen:
One word, good citizens.

First Citizen:
We are accounted poor citizens, the patricians good.
What authority surfeits on would relieve us: if they
would yield us but the superfluity, while it were
wholesome, we might guess they relieved us humanely;
but they think we are too dear: the leanness that
afflicts us, the object of our misery, is as an
inventory to particularise their abundance; our
sufferance is a gain to them Let us revenge this with
our pikes, ere we become rakes: for the gods know I
speak this in hunger for bread, not in thirst for revenge.



### We shall use character level tokenization

Later we can use other libraries/tooling

In [70]:
chars = sorted(set(text))
vocab_size = len(chars)

In [71]:
stoi = {s:i for i,s in enumerate(chars)}
itos = {i:s for i,s in enumerate(chars)}

encode = lambda s: [stoi[c] for c in s]
def decode(ids):
    return ''.join(itos[i] for i in ids)

In [72]:
tmp = "test this encoder, decoder thing"

print(encode(tmp))
print(decode(encode(tmp)))

[58, 43, 57, 58, 1, 58, 46, 47, 57, 1, 43, 52, 41, 53, 42, 43, 56, 6, 1, 42, 43, 41, 53, 42, 43, 56, 1, 58, 46, 47, 52, 45]
test this encoder, decoder thing


In [73]:
import torch

data = torch.tensor(encode(text), dtype=torch.long)
print(data.shape, data.dtype)
print(data[:1000])

torch.Size([1115394]) torch.int64
tensor([18, 47, 56, 57, 58,  1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 14, 43, 44,
        53, 56, 43,  1, 61, 43,  1, 54, 56, 53, 41, 43, 43, 42,  1, 39, 52, 63,
         1, 44, 59, 56, 58, 46, 43, 56,  6,  1, 46, 43, 39, 56,  1, 51, 43,  1,
        57, 54, 43, 39, 49,  8,  0,  0, 13, 50, 50, 10,  0, 31, 54, 43, 39, 49,
         6,  1, 57, 54, 43, 39, 49,  8,  0,  0, 18, 47, 56, 57, 58,  1, 15, 47,
        58, 47, 64, 43, 52, 10,  0, 37, 53, 59,  1, 39, 56, 43,  1, 39, 50, 50,
         1, 56, 43, 57, 53, 50, 60, 43, 42,  1, 56, 39, 58, 46, 43, 56,  1, 58,
        53,  1, 42, 47, 43,  1, 58, 46, 39, 52,  1, 58, 53,  1, 44, 39, 51, 47,
        57, 46, 12,  0,  0, 13, 50, 50, 10,  0, 30, 43, 57, 53, 50, 60, 43, 42,
         8,  1, 56, 43, 57, 53, 50, 60, 43, 42,  8,  0,  0, 18, 47, 56, 57, 58,
         1, 15, 47, 58, 47, 64, 43, 52, 10,  0, 18, 47, 56, 57, 58,  6,  1, 63,
        53, 59,  1, 49, 52, 53, 61,  1, 15, 39, 47, 59, 57,  1, 25, 39, 56, 41,
      

###  Split data

In [74]:
n = int(0.9*(len(data)))
train_data = data[:n]
val_data = data[n:]

In [75]:
block_size = 8
train_data[:block_size+1]

tensor([18, 47, 56, 57, 58,  1, 15, 47, 58])

### EXAMPLE of how we train our model, given context x whats the label y

In [76]:
x = train_data[:block_size]
y = train_data[1:block_size+1]
for t in range(block_size):
    ctx = x[:t+1]
    pred = y[t]
    print(f"given ctx: {ctx} the label is {pred}")

given ctx: tensor([18]) the label is 47
given ctx: tensor([18, 47]) the label is 56
given ctx: tensor([18, 47, 56]) the label is 57
given ctx: tensor([18, 47, 56, 57]) the label is 58
given ctx: tensor([18, 47, 56, 57, 58]) the label is 1
given ctx: tensor([18, 47, 56, 57, 58,  1]) the label is 15
given ctx: tensor([18, 47, 56, 57, 58,  1, 15]) the label is 47
given ctx: tensor([18, 47, 56, 57, 58,  1, 15, 47]) the label is 58


### These batch sizes will be computed in parallel


In [77]:
torch.manual_seed(1337)
block_size = 64
batch_size = 128

def get_batch(split):
    data = train_data if split == "train" else val_data
    idxs = torch.randint(len(data)-batch_size, (batch_size,))
    x = torch.stack([data[i:i+block_size] for i in idxs])
    y = torch.stack([data[i+1:i+block_size+1] for i in idxs])
    return x, y

xb, yb = get_batch("train")
print("-----INPUTS------")
print(f"shape: {xb.shape}")
print(f"{xb}")

print("-----OUTPUTS------")
print(f"shape: {yb.shape}")
print(f"{yb}")

print("-----EXAMPLE------")
for batch in range(batch_size):
    print(f"Batch num: {batch}")
    for idx in range(block_size):
        ctx = xb[batch,:idx+1]
        label = yb[batch, idx]
        print(f"given ctx: {ctx} , label is: {label}")

-----INPUTS------
shape: torch.Size([128, 64])
tensor([[ 1, 45, 59,  ...,  1, 44, 53],
        [52, 47, 57,  ..., 47, 43, 58],
        [50,  1, 15,  ..., 51,  1, 52],
        ...,
        [53,  1, 52,  ..., 30, 17, 19],
        [51, 39, 63,  ..., 53, 58,  1],
        [44,  1, 46,  ..., 13, 10,  0]])
-----OUTPUTS------
shape: torch.Size([128, 64])
tensor([[45, 59, 43,  ..., 44, 53, 56],
        [47, 57, 46,  ..., 43, 58,  6],
        [ 1, 15, 53,  ...,  1, 52, 53],
        ...,
        [ 1, 52, 53,  ..., 17, 19, 27],
        [39, 63,  1,  ..., 58,  1, 57],
        [ 1, 46, 43,  ..., 10,  0, 21]])
-----EXAMPLE------
Batch num: 0
given ctx: tensor([1]) , label is: 45
given ctx: tensor([ 1, 45]) , label is: 59
given ctx: tensor([ 1, 45, 59]) , label is: 43
given ctx: tensor([ 1, 45, 59, 43]) , label is: 57
given ctx: tensor([ 1, 45, 59, 43, 57]) , label is: 57
given ctx: tensor([ 1, 45, 59, 43, 57, 57]) , label is: 1
given ctx: tensor([ 1, 45, 59, 43, 57, 57,  1]) , label is: 61
given ctx:

### Learnable token embedding

### Position Encoding

In [79]:
import torch 
import torch.nn as nn 
import torch.nn.functional as F 
from torch.optim import Adam 
from torch.utils.data import TensorDataset, DataLoader 



In [80]:
class PositionEncoding(nn.Module):
    def __init__(self, d_embedding, max_len_tokens):
        super().__init__()
        # look up table of pos enc tokens
        pos_enc = torch.zeros((max_len_tokens, d_embedding))

        pos = torch.arange(start=0, end=max_len_tokens, step=1).float().unsqueeze(1)

        """
        -------------pos encodings equations---------------
        ## pos_enc(pos, 2i)   = sin(pos / 10000^(2i/d_embedding))
        ## pos_enc(pos, 2i+1) = cos(pos / 10000^(2i/d_embedding))
        """

        #0 , 2,  4, ... d_embedding represents 2i
        embedding_idx = torch.arange(start=0, end=d_embedding, step=2).float()

        div_term = 1 / torch.tensor(10000 ** (embedding_idx/d_embedding))

        pos_enc[:, 0::2] = torch.sin(pos*div_term)
        pos_enc[:, 1::2] =  torch.cos(pos*div_term)

        self.register_buffer('pe', pos_enc)

    
    def forward(self, word_embeddings):
        return word_embeddings + self.pe[:word_embeddings.size(1), :]

In [81]:
class Attention(nn.Module):
    def __init__(self, d_embeddings=32):
        super().__init__()
        self.d_embeddings = d_embeddings

        self.w_q = nn.Linear(in_features=d_embeddings, out_features=d_embeddings, bias=False)
        self.w_k = nn.Linear(in_features=d_embeddings, out_features=d_embeddings, bias=False)
        self.w_v = nn.Linear(in_features=d_embeddings, out_features=d_embeddings, bias=False)

        self.row_dim = -2
        self.col_dim = -1
    
    def forward(self, q_enc, k_enc, v_enc, mask=None):
        q ,k, v = self.w_q(q_enc), self.w_k(k_enc), self.w_v(v_enc)

        ## Compute attention scores
        ## the equation is (q * k^T)/sqrt(d_embeddings)

        similarity = torch.matmul(q, k.transpose(-2, -1))
        scaled_similarity= similarity / torch.tensor(self.d_embeddings ** 0.5)
        if mask is not None:
            scaled_similarity = scaled_similarity.masked_fill(mask=mask, value=-1e9) 
        
        attention_percents = F.softmax(scaled_similarity, dim=self.col_dim)
        attention_score = torch.matmul(attention_percents, v)

        return attention_score



In [82]:
class DecoderOnlyTransformer(nn.Module):
    def __init__(self, num_tokens = vocab_size, d_embeddings = 32, max_token_len=128):
        super().__init__()

        self.we = nn.Embedding(embedding_dim = d_embeddings,
                               num_embeddings= num_tokens,)

        self.pe = PositionEncoding(d_embedding=d_embeddings, 
                                   max_len_tokens=max_token_len)
        
        self.self_attention_1 = Attention(d_embeddings=d_embeddings)
        self.self_attention_2 = Attention(d_embeddings=d_embeddings)
        self.self_attention_3 = Attention(d_embeddings=d_embeddings)
        self.self_attention_4 = Attention(d_embeddings=d_embeddings)
        self.self_attention_5 = Attention(d_embeddings=d_embeddings)

        # d_embeddings = 32, 5 attn model head: 32*5=160 self attn values per token 

        # need a fcn to reduce dimension back to d_embeddings
        self.reduce_attention_dim = nn.Linear(in_features=d_embeddings * 5, out_features=d_embeddings)

        self.fc_layer = nn.Linear(in_features=d_embeddings, out_features=num_tokens)


    def forward(self, token_ids):
        word_embeddings = self.we(token_ids)        
        position_encoded = self.pe(word_embeddings)
        
        mask = torch.tril(torch.ones((token_ids.size(dim=-1), token_ids.size(dim=-1))))
        mask = mask == 0

        self_attention_values_1 = self.self_attention_1(position_encoded, 
                                                    position_encoded, 
                                                    position_encoded, 
                                                    mask=mask)
        
        self_attention_values_2 = self.self_attention_2(position_encoded, 
                                                    position_encoded, 
                                                    position_encoded, 
                                                    mask=mask)

        self_attention_values_3 = self.self_attention_3(position_encoded, 
                                                    position_encoded, 
                                                    position_encoded, 
                                                    mask=mask)
        self_attention_values_4 = self.self_attention_4(position_encoded, 
                                                    position_encoded, 
                                                    position_encoded, 
                                                    mask=mask)
        self_attention_values_5 = self.self_attention_5(position_encoded, 
                                                    position_encoded, 
                                                    position_encoded, 
                                                    mask=mask)
        all_self_attention_values = torch.cat([self_attention_values_1,
                                              self_attention_values_2,
                                              self_attention_values_3,
                                              self_attention_values_4,
                                              self_attention_values_5],dim=-1)

        final_self_attention_values = self.reduce_attention_dim(all_self_attention_values)

        residual_connection_values = position_encoded + final_self_attention_values

        fc_layer_output = self.fc_layer(residual_connection_values)

        return fc_layer_output



    

In [89]:
genz_model = DecoderOnlyTransformer(
                                    num_tokens=vocab_size,
                                    d_embeddings=64,
                                    max_token_len =128
                                    )

optimizer = Adam(genz_model.parameters(), lr=1e-3)
max_iters = 5000

for step in range(max_iters):
    xb, yb = get_batch("train")

    logits = genz_model(xb) # B, T, Vocab size

    loss = F.cross_entropy(logits.view(-1, vocab_size), yb.view(-1)) # expects (N, num_classes) 


    if step %100 == 0:
        xv, yv = get_batch("val")
        logits_v = genz_model(xv)
        loss_v = F.cross_entropy(logits_v.view(-1, vocab_size), yv.view(-1))
        print(f"step {step}: train loss: {loss.item():.4f}, val_loss: {loss_v.item():.4f}")
    optimizer.zero_grad()
    loss.backward()
    optimizer.step()
    




/tmp/ipykernel_47167/3736413204.py:18: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.detach().clone() or sourceTensor.detach().clone().requires_grad_(True), rather than torch.tensor(sourceTensor).
  div_term = 1 / torch.tensor(10000 ** (embedding_idx/d_embedding))


step 0: train loss: 4.4544, val_loss: 4.4656
step 100: train loss: 2.5917, val_loss: 2.5932
step 200: train loss: 2.4076, val_loss: 2.4550
step 300: train loss: 2.2996, val_loss: 2.3675
step 400: train loss: 2.2398, val_loss: 2.2797
step 500: train loss: 2.1771, val_loss: 2.2192
step 600: train loss: 2.1596, val_loss: 2.1947
step 700: train loss: 2.1192, val_loss: 2.1931
step 800: train loss: 2.0803, val_loss: 2.1149
step 900: train loss: 2.0531, val_loss: 2.1237
step 1000: train loss: 2.0058, val_loss: 2.1063
step 1100: train loss: 2.0469, val_loss: 2.1304
step 1200: train loss: 2.0020, val_loss: 2.1253
step 1300: train loss: 1.9512, val_loss: 2.1009
step 1400: train loss: 1.9929, val_loss: 2.0781
step 1500: train loss: 1.9242, val_loss: 2.0738
step 1600: train loss: 1.9110, val_loss: 2.0671
step 1700: train loss: 1.9127, val_loss: 2.0469
step 1800: train loss: 1.9285, val_loss: 2.1161
step 1900: train loss: 1.9682, val_loss: 2.0708
step 2000: train loss: 1.9392, val_loss: 2.0639
step

In [90]:
def generate(max_new_tokens, seed):
    idx = torch.tensor(encode(seed), dtype=torch.long).unsqueeze(0)

    for _ in range(max_new_tokens):
        # Only keep the last block_size tokens
        idx_cond = idx[:, -block_size:]

        # Get predictions
        logits = genz_model(idx_cond)

        # Only care about the final token's predictions
        last_logits = logits[:, -1, :]

        # Convert logits to  probabilities
        probs = F.softmax(last_logits, dim=-1)

        # Sample next token
        next_token = torch.multinomial(probs, num_samples=1)

        # Add it to our sequence
        idx = torch.cat([idx, next_token], dim=1)

    return decode(idx[0].tolist())
    

In [85]:
genz_model.eval()

while True:
    user_input = input("You: ")

    if user_input.lower() == "quit":
        break

    response = generate(100, user_input)

    print("GenZGPT:", response)

GenZGPT: hither gecomain I spumsin, and flience a be stien what misty bed, loveced:
Do noth, chosin, looks orseappa
GenZGPT: what thou name, to briarugh tay;
Whostence sofe of thiss!
Mesin that as such olike
othrace, in feade anconour'cery


In [91]:
print(generate(max_new_tokens=2000, seed=" "))


 muparinsm rear exce? what
fience themes thee it
Abl'd for you have wroung from not now.

BARUTIUS:
That tor rut.
And at me not thy ance to throk answer's acy, and Sancy,
What me woul, be too freave fle me boure do rever Laway?

GLOUCESTER:
But I bnooughnds, for mead
meo on is uall fitd he in leancllh I wouth clow came heat and May life,
that tind.
'Citize, sife.--


GLOUCESTER:
Siday chase bather I last nouap, 'erciany;
That hersho; and fruch the wre crayet
oo' the wart. O, the her, idrince:
Were my the allar's come till pity, boy all man
ellos. Henry weret, did what? thinat but afe Hert warm?

AUTOLYCUS:
Nay, whaty too my stiend what
chine havenge, nowd maind you self.
Thish moth, wagyted wor is covet's of so oubsecan theof bee ot the and how for him full Sance too not pare and thee purpoep-kin but treatenk's thoke;
Neur, as good ukned rat I can blue,
o not, longe! wherembe guince re' Lod Hernit ost,
On mechome trebrooknence! willaced vise,
To by this thee be toose affath, Pagiom
To 